In [5]:
from dotenv import load_dotenv
import os

load_dotenv()
NEON_CONN_STRING = os.getenv("NEON_CONN_STRING")
NEON_TEST_CONN_STRING = os.getenv("NEON_TEST_CONN_STRING")
print("Test branch connection loaded:", NEON_TEST_CONN_STRING is not None)

Test branch connection loaded: True


In [3]:
import subprocess

backup_file = "../backups/smakjit_backup_20260719_121724.dump"

result = subprocess.run(
    [
        "pg_restore",
        "--dbname=" + NEON_TEST_CONN_STRING,
        "--clean",
        "--if-exists",
        "--verbose",
        backup_file
    ],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print("STDOUT:", result.stdout[-2000:])
print("STDERR:", result.stderr[-2000:])

Return code: 1
STDOUT: 
STDERR:  creating INDEX "public.opportunity_status_idx"
pg_restore: creating FK CONSTRAINT "public.ApplicationAnswer ApplicationAnswer_application_id_fkey"
pg_restore: creating FK CONSTRAINT "public.Application Application_opp_id_fkey"
pg_restore: creating FK CONSTRAINT "public.Application Application_user_id_fkey"
pg_restore: creating FK CONSTRAINT "public.OpportunitySkill OpportunitySkill_opp_id_fkey"
pg_restore: creating FK CONSTRAINT "public.OpportunitySkill OpportunitySkill_skill_id_fkey"
pg_restore: creating FK CONSTRAINT "public.Opportunity Opportunity_category_id_fkey"
pg_restore: creating FK CONSTRAINT "public.Opportunity Opportunity_org_id_fkey"
pg_restore: creating FK CONSTRAINT "public.Organization Organization_owner_id_fkey"
pg_restore: creating FK CONSTRAINT "public.Organization Organization_reviewed_by_fkey"
pg_restore: creating FK CONSTRAINT "public.saved_opportunity SavedOpportunity_opp_id_fkey"
pg_restore: creating FK CONSTRAINT "public.saved_o

In [7]:
import psycopg2

def get_counts(conn_string, tables):
    conn = psycopg2.connect(conn_string)
    cur = conn.cursor()
    counts = {}
    for table in tables:
        cur.execute(f'SELECT COUNT(*) FROM "{table}"')
        counts[table] = cur.fetchone()[0]
    cur.close()
    conn.close()
    return counts

tables_to_check = ["users", "Organization", "Opportunity", "Application", "VolunteerProfile"]

prod_counts = get_counts(NEON_CONN_STRING, tables_to_check)
restored_counts = get_counts(NEON_TEST_CONN_STRING, tables_to_check)

print(f"{'Table':<20} {'Production':<12} {'Restored':<12} {'Match'}")
for t in tables_to_check:
    match = "MATCH" if prod_counts[t] == restored_counts[t] else "MISMATCH"
    print(f"{t:<20} {prod_counts[t]:<12} {restored_counts[t]:<12} {match}")

Table                Production   Restored     Match
users                42           42           MATCH
Organization         10           10           MATCH
Opportunity          60           60           MATCH
Application          61           61           MATCH
VolunteerProfile     26           26           MATCH
